# L10 · RLHF-PPO End to End

## Goal

- connect SFT, reward, rollout, and update
- separate policy and frozen-model ownership
- decompose reward and KL

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L10:toy:42").hexdigest()
print(f"lesson=L10 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L10 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:7152104ceff29985d0262d3436bb57ad231f9f0d5822839b8bb374016cc72d1d data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: LLM policy → **end-to-end RLHF-PPO** → comparison/evaluation

$$\text{SFT}\rightarrow\text{preference/RM}\rightarrow\text{rollout}\rightarrow\text{reward+KL}\rightarrow\text{PPO update}$$

RLHF-PPO separates trainable policy/value from frozen reference/reward models. This toy uses a deterministic verifier for a fully offline pipeline, while preserving the ownership and mask contracts used with real models.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** At PPO start, which SFT-policy hash should the reference hash match? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>It must match the initial SFT policy copied before rollout and remain frozen afterward.</details>

In [2]:
from rl_study.algorithms.rlhf_ppo import train_rlhf_ppo
from rl_study.algorithms.sft import train_sft
from rl_study.models.roles import parameter_sha256
sft_stage = train_sft(steps=2, batch_size=4, seed=42)
sft_hash = parameter_sha256(sft_stage.model)
rlhf_stage = train_rlhf_ppo(
    updates=1, batch_size=2, seed=42, policy=sft_stage.model,
    reward_source="verifier", update_epochs=1
)
print({"sft_steps": 2, "rlhf_updates": 1,
       "initial_sft_hash": sft_hash[:20],
       "generated_tokens": rlhf_stage.generated_tokens,
       "reference_hash": rlhf_stage.reference_hash[:20],
       "policy_loss": round(rlhf_stage.policy_losses[-1], 4)})

{'sft_steps': 2, 'rlhf_updates': 1, 'initial_sft_hash': 'sha256:6567e5680f133', 'generated_tokens': 44, 'reference_hash': 'sha256:6567e5680f133', 'policy_loss': 0.0}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Public stage APIs expose the lifecycle without copying trainer internals into the notebook. A learned reward model is an alternative and shares the verifier interface in the C5 implementation.

**Common trap:** If the reference updates with the policy, the KL anchor moves and the penalty appears artificially small. Audit its hash, `requires_grad=False`, and optimizer membership. Regression tests: `test_rlhf_ppo_ratio_one_and_gradient_ownership`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert rlhf_stage.generated_tokens > 0
assert rlhf_stage.reference_hash == sft_hash
print("checks=passed")

checks=passed


**Recall:** Which of policy, value, reference, and reward roles should be optimizer-owned? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** After two SFT steps, the run generated 44 tokens and the reference hash matched the initial SFT hash. A one-update policy loss of 0.0 reflects the ratio-1 starting point, not training success.
- Executable checks: `test_rlhf_ppo_ratio_one_and_gradient_ownership`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L11 optimizes the policy directly from chosen/rejected pairs without online rollouts.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `learning-to-summarize-2020` — `docs/sources.yml`
- `instructgpt-2022` — `docs/sources.yml`
- `repo-summarize-from-feedback` — `docs/sources.yml`